# PT & OT Consult Orders
Creates CLIF *key_icu_orders* table from MIMIC data.

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from datetime import datetime
from datetime import timedelta
import numpy as np
import os
import json

#Load file path
with open('../config/config.json', 'r') as file:
    config = json.load(file)
path = path = os.path.join(config['mimic'], 'icu', 'chartevents.parquet')
pyarr_file = pq.ParquetFile(path)

# Define chunk size (adjust based on available memory)
CHUNK_SIZE = 10000

mapping = {
    "PT": "pt_evaluation",
    "OT": "ot_evaluation"
}

processed_chunks = []

for batch in pyarr_file.iter_batches(batch_size=CHUNK_SIZE):
    chunk = batch.to_pandas()
    
    # Filter for PT orders only
    chart_mask = (
        chunk["hadm_id"].notna() &
        chunk["itemid"].notna() &
        (chunk["itemid"] == 225135) &
        chunk["value"].notna() &
        chunk["value"].isin(["PT", "OT"])
    )
    chunk = chunk[chart_mask]

    # Skip empty chunks
    if chunk.empty:
        continue

    # Convert string data types
    chunk['hospitalization_id'] = chunk['hadm_id'].astype(str)
    chunk['order_status'] = 'sent'

    # Convert times
    chunk['order_dttm'] = pd.to_datetime(chunk['charttime'], utc=False)
    chunk['order_dttm'] = chunk['order_dttm'].dt.tz_localize(
        'America/New_York', ambiguous=True, nonexistent='shift_forward'
    )
    chunk['order_dttm'] = chunk['order_dttm'].dt.tz_convert('UTC')

    # Convert order name and categorize
    chunk['order_name'] = chunk['value']
    chunk['order_category'] = chunk['order_name'].map(mapping)

    processed_chunks.append(chunk)

# Combine all chunks
pt_df = pd.concat(processed_chunks, ignore_index=True)

#Reorganize columns
pt_df = pt_df[['hospitalization_id','order_dttm','order_name','order_category','order_status']]
pt_df.dtypes

/tmp/ipykernel_595143/2294962528.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['order_dttm'] = pd.to_datetime(chunk['charttime'], utc=False)
/tmp/ipykernel_595143/2294962528.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['order_dttm'] = pd.to_datetime(chunk['charttime'], utc=False)
/tmp/ipykernel_595143/2294962528.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  chunk['order_dttm'] = pd.to_datetime(chunk['charttime'], utc=False)
/tmp/ipykernel_595143/2294962528.py:47: UserWarning: Could not infer format, so each element will be parsed individual

hospitalization_id                 object
order_dttm            datetime64[ns, UTC]
order_name                         object
order_category                     object
order_status                       object
dtype: object

In [2]:
#Save
import os
path = os.path.join(config['output_folder'], "clif_key_icu_orders.parquet")
pt_df.to_parquet(path)

In [3]:
#Sample
pt_df.head()

,hospitalization_id,order_dttm,order_name,order_category,order_status
0,26184834,2131-01-11 23:37:00+00:00,OT,ot_evaluation,sent
1,26184834,2131-01-11 23:37:00+00:00,PT,pt_evaluation,sent
2,26184834,2131-01-11 23:54:00+00:00,OT,ot_evaluation,sent
3,26184834,2131-01-11 23:54:00+00:00,PT,pt_evaluation,sent
4,22725460,2112-12-05 10:23:00+00:00,OT,ot_evaluation,sent


In [4]:
# Post-processing diagnostics
print(f"Shape: {pt_df.shape}")
print(f"Blank order times: {pt_df['order_dttm'].isna().sum()}")
print(pt_df['order_category'].value_counts())

Shape: (40996, 5)
Blank order times: 0
order_category
pt_evaluation    29203
ot_evaluation    11793
Name: count, dtype: int64
